# Multivariable Calculus

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 02.07 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/08_multivariable_calculus.ipynb)

---

## 🎯 Learning Objective

Extend integration to multiple variables with double and triple integrals, and learn how change-of-variables and Fubini's theorem simplify these computations.

---

## 📐 Theory

`02.07` integrated a single-variable function to get a signed area. This notebook extends that
to functions of two or three variables, where the integral measures **volume** (or a
higher-dimensional analogue) instead.

### Double integrals

For $f(x,y)$ over a region $R$, the double integral
$\iint_R f(x,y)\,dA$ is the signed volume between the surface $z=f(x,y)$ and the $xy$-plane,
built the same way as before: partition $R$ into tiny cells of area $\Delta A$, sum
$f(x_i,y_i)\Delta A$ over all cells, and take the limit as the cells shrink to points.

### Fubini's theorem: reducing to iterated 1D integrals

Computing a genuine 2D limit-of-sums is unwieldy. **Fubini's theorem** says that (for
well-behaved $f$) a double integral over a rectangle $[a,b]\times[c,d]$ can be computed as two
ordinary 1D integrals, nested:

$$\iint_{[a,b]\times[c,d]} f(x,y)\,dA = \int_a^b\left(\int_c^d f(x,y)\,dy\right)dx =
\int_c^d\left(\int_a^b f(x,y)\,dx\right)dy$$

Integrate with respect to one variable first (treating the other as fixed, exactly like a
partial derivative in `02.03` but for integration), then integrate the resulting function of
the remaining variable. Fubini's theorem guarantees you get the *same answer* regardless of
which variable you integrate first — this order-independence is the 2D integration analogue of
Clairaut's theorem for mixed partials.

### Triple integrals

The same idea extends to $\iiint_V f(x,y,z)\,dV$ over a 3D region $V$, computing a
4-dimensional "hypervolume" when $f$ isn't constant, or ordinary volume when $f\equiv 1$. Fubini
again lets you peel it apart into three nested 1D integrals.

### Change of variables

Sometimes a region or integrand is awkward in Cartesian coordinates $(x,y)$ but simple in
another coordinate system, like polar $(r,\theta)$. The **change of variables formula** lets
you substitute:

$$\iint_R f(x,y)\,dx\,dy = \iint_{R'} f(x(r,\theta), y(r,\theta))\, |J|\, dr\,d\theta$$

where $|J|$ is the absolute value of the determinant of the Jacobian (recall `02.06`) of the
coordinate transformation — for polar coordinates, $|J| = r$. This $|J|$ factor is a
*stretching correction*: transforming coordinates distorts area/volume locally, and the
Jacobian determinant is exactly the local scaling factor needed to compensate, so the total
integral still measures the same true volume regardless of which coordinates you compute in.

### Why this matters here

Once you reach probability (Module `03`), a joint density $p(x,y)$ over two random variables
must integrate to $1$ over the *whole plane* — a double integral. Recovering the marginal
distribution of just $x$ means integrating $y$ out: $p(x) = \int p(x,y)\,dy$ — a Fubini-style
reduction that appears constantly under the name **marginalization**. Normalizing constants in
Bayesian inference (`03.05`) are exactly this kind of integral, often over many more than two
variables, which is why sampling methods (`03.09`) exist: for high-dimensional integrals,
direct evaluation is often intractable and Monte Carlo estimation becomes the practical tool.

---

In [ ]:
# Setup
import numpy as np
import sympy as sp
import scipy.integrate as spi
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A 3D surface plot of $f(x,y)$ over a rectangular region, with the volume between the surface
and the plane shaded, makes the double integral's "volume under a surface" meaning concrete.

In [ ]:
# f(x,y) = exp(-(x^2+y^2)) over [-2,2]x[-2,2] -- a bump whose double integral is the volume underneath
f = lambda x, y: np.exp(-(x**2 + y**2))
x = np.linspace(-2, 2, 60)
y = np.linspace(-2, 2, 60)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap="viridis", alpha=0.9)
ax1.set_xlabel("x"); ax1.set_ylabel("y"); ax1.set_zlabel("f(x,y)")
ax1.set_title("Surface z=exp(-(x^2+y^2)):\ndouble integral = volume underneath")

def trapz_1d(values, points):
    # Same trapezoidal rule as 02.07's trapezoidal_rule, written for arbitrary sample arrays
    # (avoids relying on a numpy version-specific trapz/trapezoid name)
    widths = np.diff(points)
    return np.sum(widths * (values[:-1] + values[1:]) / 2)

# Riemann-sum-style volume estimate: sum f(x,y) * cell_area over a fine grid (2D generalization
# of the 1D rectangle sums from 02.07)
cell_area = (x[1] - x[0]) * (y[1] - y[0])
riemann_volume = np.sum(Z) * cell_area

# Fubini check: integrate over y first (for each x), then integrate that result over x
inner_integral = np.array([trapz_1d(Z[:, i], y) for i in range(len(x))])  # integrate along y, per x
fubini_volume = trapz_1d(inner_integral, x)  # integrate the result along x -- outer integral

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(x, inner_integral, color="#4C72B0", lw=2)
ax2.set_xlabel("x"); ax2.set_ylabel(r"$\int f(x,y)\,dy$ (inner integral, for each x)")
ax2.set_title("Fubini's theorem: integrate y out first,\nthen integrate the resulting 1D function")
plt.tight_layout()
plt.show()

exact = np.pi * (1 - np.exp(-4))**1  # approx correction factor for the finite domain vs full plane
print(f"Riemann-sum volume estimate:         {riemann_volume:.5f}")
print(f"Fubini (iterated 1D integrals) volume: {fubini_volume:.5f}")
print(f"(As the domain grows to cover the whole plane, this approaches pi = {np.pi:.5f} exactly --")
print(f" the classic Gaussian integral result used throughout probability, Module 03.)")

## 🐍 Implementation from Scratch

We implement a 2D Riemann sum directly, verify Fubini's theorem by integrating in both orders,
verify the change-of-variables formula on a polar-coordinates example, and cross-check
everything against `scipy.integrate.dblquad`.

In [ ]:
def double_integral_riemann(f, x_range, y_range, n=200):
    """2D Riemann sum: sum f(x,y) over a fine grid, weighted by cell area -- the direct
    generalization of the 1D rectangle sums from 02.07."""
    xs = np.linspace(*x_range, n)
    ys = np.linspace(*y_range, n)
    X, Y = np.meshgrid(xs, ys)
    cell_area = (xs[1] - xs[0]) * (ys[1] - ys[0])
    return np.sum(f(X, Y)) * cell_area

# Fubini check on f(x,y) = x*y^2 over [0,2]x[0,3] -- has a clean closed form to check against
f2 = lambda x, y: x * y**2
riemann_est = double_integral_riemann(f2, (0, 2), (0, 3), n=400)
# NOTE: scipy's dblquad calls func(y, x) -- y first -- so we swap the argument order here
scipy_est, _ = spi.dblquad(lambda y, x: f2(x, y), 0, 2, 0, 3)
x_sym, y_sym = sp.symbols('x y')
exact_sym = sp.integrate(sp.integrate(x_sym * y_sym**2, (y_sym, 0, 3)), (x_sym, 0, 2))

print(f"Riemann sum estimate:        {riemann_est:.5f}")
print(f"scipy.integrate.dblquad:     {scipy_est:.5f}")
print(f"Exact (SymPy, iterated):     {float(exact_sym):.5f}")

# Change of variables: integrate a disk of radius R using polar coordinates, where the
# Jacobian determinant |J| = r makes the substitution correct
R = 2.0
g_polar = lambda r, theta: (r * np.cos(theta))**2 * r  # g(x,y)=x^2, x=r*cos(theta), times |J|=r
polar_est, _ = spi.dblquad(g_polar, 0, 2 * np.pi, 0, R)
# Cross-check directly in Cartesian coordinates using a masked Riemann sum over the same disk
xs = np.linspace(-R, R, 800); ys = np.linspace(-R, R, 800)
Xc, Yc = np.meshgrid(xs, ys)
mask = Xc**2 + Yc**2 <= R**2
cell_area = (xs[1]-xs[0]) * (ys[1]-ys[0])
cartesian_est = np.sum((Xc**2)[mask]) * cell_area

print(f"\nIntegral of x^2 over a disk of radius {R} -- polar (with Jacobian |J|=r): {polar_est:.5f}")
print(f"Same integral estimated directly in Cartesian coordinates:              {cartesian_est:.5f}")
print(f"Agreement confirms the |J|=r correction factor is doing its job.")

## 🤖 Why This Matters for AI

**Marginalization** — recovering $p(x)$ from a joint density $p(x,y)$ by integrating $y$ out —
is exactly the Fubini-style "integrate one variable, holding the rest" operation from this
notebook, and it's everywhere in probabilistic ML: a Gaussian Mixture Model's marginal
likelihood, a VAE's evidence $p(x) = \int p(x|z)p(z)\,dz$, and a Bayesian model's posterior
predictive distribution are all multivariable integrals over latent variables. When these
integrals have no closed form (routine once distributions get complex), Module `03.09`'s
Monte Carlo methods approximate them the same way our Riemann sum approximated a double
integral above: sample points, weight, and average. Below, we compute a 2D Gaussian's
marginal distribution two ways — via the closed-form Gaussian integral, and via a numeric
double-integral marginalization — and confirm they agree.

In [ ]:
from scipy.stats import norm

# A correlated bivariate Gaussian joint density p(x, y) -- a stand-in for a joint distribution
# over, say, a latent variable y and an observation x in a probabilistic model
rho = 0.6  # correlation between x and y
def joint_pdf(x, y):
    return (1 / (2*np.pi*np.sqrt(1-rho**2))) * np.exp(-(x**2 - 2*rho*x*y + y**2) / (2*(1-rho**2)))

# Marginalize y out numerically: p(x) = integral of p(x,y) dy -- exactly the Fubini-style
# "integrate one variable, hold the other fixed" operation from the Theory section
y_grid = np.linspace(-8, 8, 4000)
x_test_points = np.array([-1.5, 0.0, 0.8, 2.0])
marginal_numeric = np.array([trapz_1d(joint_pdf(xv, y_grid), y_grid) for xv in x_test_points])

# Closed form: for THIS joint density, marginalizing out y always recovers a standard normal
# in x -- a known property of the bivariate Gaussian, giving us an exact answer to check against
marginal_exact = norm.pdf(x_test_points, loc=0, scale=1)

print("Marginalizing a joint density by numeric integration (Fubini in action):")
for xv, num, exact in zip(x_test_points, marginal_numeric, marginal_exact):
    print(f"  p(x={xv:+.1f}) via double-integral marginalization: {num:.6f}   exact: {exact:.6f}")
print(f"\nAll match: {np.allclose(marginal_numeric, marginal_exact, atol=1e-4)}")

fig, ax = plt.subplots(figsize=(7, 4.5))
x_fine = np.linspace(-4, 4, 200)
marginal_fine = np.array([trapz_1d(joint_pdf(xv, y_grid), y_grid) for xv in x_fine])
ax.plot(x_fine, marginal_fine, color="#4C72B0", lw=3, label="marginalized numerically (this notebook)")
ax.plot(x_fine, norm.pdf(x_fine), color="#DD8452", lw=1.5, ls="--", label="exact standard normal")
ax.set_xlabel("x"); ax.set_ylabel("p(x)")
ax.set_title("Marginalization: integrating a latent variable out of a joint density")
ax.legend()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Compute $\iint_{[0,1]\times[0,2]} (2x + y)\,dA$ by hand using Fubini's theorem, integrating in
   both orders and confirming they agree. Verify with `double_integral_riemann`, and as a
   sanity check confirm the numeric estimate is positive — the integrand $2x+y$ is $\ge 0$
   everywhere on this region, so a signed double integral over it can never come out negative.

<details>
<summary>💡 Solution</summary>

Integrating $y$ first: $\int_0^2(2x+y)\,dy = \left[2xy+\frac{y^2}{2}\right]_0^2=4x+2$, then
$\int_0^1(4x+2)\,dx=\left[2x^2+2x\right]_0^1=4$. Integrating $x$ first: $\int_0^1(2x+y)\,dx=
\left[x^2+xy\right]_0^1=1+y$, then $\int_0^2(1+y)\,dy=\left[y+\frac{y^2}{2}\right]_0^2=4$ —
identical, exactly as Fubini's theorem guarantees. `double_integral_riemann(f2, (0,1), (0,2))`
(the notebook's default $n=200$) gives $\approx4.04$, and refining to $n=400$ tightens it to
$\approx4.02$ — both converging toward the exact value $4$ as the grid gets finer, with the
positive integrand ($2x+y\ge0$ throughout $[0,1]\times[0,2]$) correctly producing a positive
estimate at every resolution, never a spurious negative one.

</details>

### 💭 UNDERSTAND
2. A learner notices $\iint_{[0,1]\times[0,2]} xy\,dA$ can be computed as the *product* of two
   separate 1D integrals, $\left(\int_0^1x\,dx\right)\left(\int_0^2y\,dy\right)$, and gets the
   right answer. They then try the same shortcut on $\iint_{[0,1]\times[0,2]} (x+y)\,dA$,
   computing $\left(\int_0^1x\,dx\right)\left(\int_0^2y\,dy\right)$ again. Explain why the
   shortcut is valid for $xy$ but not for $x+y$, and show the two integrals actually disagree
   for $x+y$.

<details>
<summary>💡 Solution</summary>

The product shortcut $\iint f(x)g(y)\,dA = \left(\int f(x)\,dx\right)\left(\int g(y)\,dy\right)$
only holds when the integrand **factors** into a pure function of $x$ times a pure function of
$y$ — exactly what $xy$ does ($f(x)=x$, $g(y)=y$), which is why
$\left(\int_0^1x\,dx\right)\left(\int_0^2y\,dy\right)=\left(\frac12\right)(2)=1$ correctly
matches the true double integral of $xy$. The function $x+y$ does **not** factor this way (there
is no pair of single-variable functions whose *product* equals $x+y$), so the shortcut simply
doesn't apply — Fubini's theorem still lets you compute $\iint(x+y)\,dA$ as *nested* integrals
(integrate one variable while holding the other fixed, then integrate the result), but nested
integration of a sum is not the same operation as a product of separate integrals. Carrying out
Fubini correctly: $\int_0^1\left(\int_0^2(x+y)\,dy\right)dx = \int_0^1(2x+2)\,dx=3$ — while the
learner's misapplied product shortcut gives $\left(\int_0^1x\,dx\right)\left(\int_0^2y\,dy\right)
=\left(\frac12\right)(2)=1$, a different (and wrong) number. The two only happen to agree for
integrands that are genuinely a product of one function of $x$ and one function of $y$.

</details>

### ✏️ DERIVE
3. Derive the Jacobian determinant for the polar coordinate transformation $x=r\cos\theta$,
   $y=r\sin\theta$: write the $2\times2$ Jacobian matrix of partial derivatives, compute its
   determinant symbolically, and confirm it equals $r$ (the factor $|J|=r$ the Theory section
   uses without derivation). Verify your symbolic result matches a numeric finite-difference
   Jacobian at several $(r,\theta)$ points.

<details>
<summary>💡 Solution outline</summary>

$$J=\begin{pmatrix}\partial x/\partial r & \partial x/\partial\theta\\
\partial y/\partial r & \partial y/\partial\theta\end{pmatrix}=
\begin{pmatrix}\cos\theta & -r\sin\theta\\ \sin\theta & r\cos\theta\end{pmatrix}.$$
The determinant is $\det J = (\cos\theta)(r\cos\theta)-(-r\sin\theta)(\sin\theta) =
r\cos^2\theta+r\sin^2\theta = r(\cos^2\theta+\sin^2\theta)=r$, using the Pythagorean identity.
So $|J|=r$ (already non-negative for $r\ge0$, so no extra absolute value is needed) — exactly
the stretching-correction factor asserted in the Theory section. A finite-difference check at
several points (e.g. $r=0.5,1.5,3.0$ crossed with $\theta=0,\pi/4,\pi/2,\pi$) reproduces
$\det J = r$ numerically to 6 decimal places at every single point tried, confirming the factor
isn't specific to any one location — it scales linearly with $r$ everywhere, which is also the
intuitive reason polar area elements grow with distance from the origin: a fixed angular wedge
$d\theta$ sweeps out more physical area the farther out $r$ is.

</details>

### 🐍 IMPLEMENT
4. Implement `polar_double_integral(g, r_max, n_r, n_theta)`: a double Riemann sum directly in
   polar coordinates (sum $g(r\cos\theta, r\sin\theta)\cdot r$ over a grid of $(r,\theta)$
   values, weighted by $\Delta r\,\Delta\theta$ — i.e. build the $|J|=r$ factor in yourself
   rather than converting to Cartesian first). Test it by reproducing the Visual Intuition
   section's $\iint_D e^{-(x^2+y^2)}\,dA$ over the disk of radius 3, checking against both the
   closed form $\pi(1-e^{-9})$ and a masked Cartesian `double_integral_riemann`-style sum over
   the same disk.

<details>
<summary>✅ How to know you're right</summary>

All three should agree closely: the closed form is $\pi(1-e^{-9})\approx3.14120495$; a masked
Cartesian Riemann sum over the disk (fine grid, e.g. $1200\times1200$) should land within about
$3\times10^{-7}$ of that; and `polar_double_integral` (e.g. $n_r=n_\theta=400$) should land
within about $2\times10^{-5}$ of it as well — all three within roughly $0.001\%$ of each other,
comfortably inside a $1\%$ tolerance. If your polar version disagrees by much more than that,
the most common bug is forgetting to multiply the integrand by $r$ (the $|J|=r$ factor from
DERIVE) before summing — without it, you're computing $\iint g\,dr\,d\theta$, a different
(and dimensionally wrong) quantity, not $\iint g\,dA$.

</details>

### 🤖 APPLY
5. Build a 2-component **mixture** joint density $p(x,y) = 0.5\,\mathcal{N}(x;-1,1)\mathcal{N}(y;0,1)
   + 0.5\,\mathcal{N}(x;2,1)\mathcal{N}(y;0,1)$ (component index affects the mean of $x$ only;
   $y$ is always standard normal, independent of the component). Using the same numeric
   marginalization technique as the AI section, compute the marginal $p(x)$ at several points
   and the marginal $p(y)$ at several points, and identify which one comes out as a genuine
   two-mode mixture and which comes out as a single, ordinary Gaussian — then explain why,
   structurally, it has to work out that way for *this* joint density.

<details>
<summary>✅ What you should observe</summary>

Numerically marginalizing out $y$ (integrating $p(x,y)$ over $y$ for fixed $x$) should reproduce
the two-component mixture $0.5\,\mathcal{N}(x;-1,1)+0.5\,\mathcal{N}(x;2,1)$ almost exactly (in
one run, matching to 4+ decimal places at $x=-1,2,0.5$) — a genuinely bimodal-shaped density.
Numerically marginalizing out $x$ (integrating over $x$ for fixed $y$) should instead reproduce
a single, plain $\mathcal{N}(0,1)$ exactly (matching to 4+ decimal places at $y=0,1$) — no trace
of two modes at all. Structurally this has to happen because $y$'s distribution doesn't depend
on which component is active — integrating $p(x,y)$ over $x$ pulls the mixture weights
($0.5+0.5=1$) out from under the $y$-factor entirely, collapsing the sum back to a single
Gaussian in $y$ regardless of the $x$-side's structure — while integrating over $y$ leaves the
$x$-side's two means and mixture weights fully intact. This is exactly the same "marginals lose
information the joint had" phenomenon the notebook's own bivariate-Gaussian example shows for
correlation, but now demonstrated on a genuinely multi-modal joint rather than a unimodal one.

</details>

### 🚀 CHALLENGE
6. Extend this notebook's double-integral machinery to a **triple** integral: compute the
   volume of the solid bounded above by the paraboloid $z=4-x^2-y^2$ and below by $z=0$ (its
   footprint is the disk $x^2+y^2\le4$, where the paraboloid is non-negative), using at least
   two different methods among: (a) reducing to a 2D double integral of the height function
   $4-x^2-y^2$ over the disk (reusing this notebook's own 2D tools), (b) a genuine 3D
   voxel-counting Riemann sum, and (c) `scipy.integrate.tplquad`. Compare all methods against
   the closed-form answer $8\pi$ (obtainable via a polar substitution in the double integral),
   and discuss how the cost of each numerical method scales as you add a 4th dimension.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms all methods agree with $8\pi\approx25.1327$: reducing to a 2D double
integral of the height function over the disk should land within about $0.001\%$
($\approx25.1327$ at a few-hundred-point grid), `scipy.integrate.tplquad` should match to
several decimal places ($\approx25.13274$, with a tiny reported error estimate), and a direct
3D voxel-counting Riemann sum should land within roughly $0.3\%$ at a comparable resolution
(coarser than the other two because it discretizes one more dimension at the same per-axis
grid density). A strong answer also derives or at least cites the closed form: converting to
polar, $\int_0^{2\pi}\int_0^2(4-r^2)\,r\,dr\,d\theta = 2\pi\cdot4=8\pi$ (inner integral
$\int_0^2(4r-r^3)\,dr=8-4=4$). For the scaling question, a strong answer notes that a
fixed-resolution grid with $m$ points per axis costs $O(m^3)$ cells for a volume integral but
$O(m^4)$ for a 4D hypervolume integral at the *same* per-axis resolution — at $m=300$ that's
$2.7\times10^7$ cells growing to $8.1\times10^9$, a $300\times$ jump from adding just one more
dimension — the same curse-of-dimensionality wall this notebook's own "Why This Matters for AI"
section names as the reason Module `03.09`'s Monte Carlo methods exist: Monte Carlo's cost
scales with the *number of samples*, not with (grid points)$^{\text{dimension}}$, which is why
it becomes the practical choice once a fixed grid's cost becomes intractable.

</details>

---

## 📚 Further Reading
- Stewart, *Calculus: Early Transcendentals*, Ch. 15 (Multiple Integrals)
- Bishop, *Pattern Recognition and Machine Learning*, Ch. 2.3 (The Gaussian Distribution)

---

*Next notebook: [Vector Calculus](09_vector_calculus.ipynb)*